In [1]:
from data.uniner_utils import load_uniner_dataset
from data.adapters import uniner_to_gliner
from data.chunking import chunk_gliner_dataset
from data.split_utils import split_dataset
from training.train_utils import load_gliner_model, train_gliner_model
import torch

# 1. Load UniNER dataset
uniner_dataset = load_uniner_dataset()
model = load_gliner_model()
tokenizer = model.data_processor.transformer_tokenizer

/home/rfgch/proyectos/GlinerTrain/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]/home/rfgch/proyectos/GlinerTrain/env/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 116.63it/s]
/home/rfgch/proyectos/GlinerTrain/env/lib/python3.12/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means 

In [2]:
# 2. Split (70/20/10 por defecto)
train_uniner_data, val_uniner_data, test_uniner_data = split_dataset(uniner_dataset)

# 3. Adapt UniNER → GLiNER format
train_uniner_data = uniner_to_gliner(train_uniner_data, tokenizer)
val_uniner_data   = uniner_to_gliner(val_uniner_data, tokenizer)
test_uniner_data  = uniner_to_gliner(test_uniner_data, tokenizer)

# 4. Chunking (512 tokens)
train_uniner_data = chunk_gliner_dataset(train_uniner_data, max_length=512)
val_uniner_data   = chunk_gliner_dataset(val_uniner_data, max_length=512)
test_uniner_data  = chunk_gliner_dataset(test_uniner_data, max_length=512)

Eval Base Model

In [ ]:
model.eval()
with torch.no_grad():
    metrics = model.evaluate(
        test_uniner_data,
        batch_size=8
    )

print(metrics)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


('P: 55.50%\tR: 31.18%\tF1: 39.93%\n', np.float64(0.3993206440612747))


Train Model

In [ ]:
# 5. Train
train_gliner_model(
    model=model,
    train_data=train_uniner_data,
    eval_data=val_uniner_data,
    output_dir="./models/gliner_uniner",
    max_steps=100
)

/home/rfgch/projects/GlinerTrain/gliner-env/lib/python3.11/site-packages/gliner/model.py:1093: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Step,Training Loss
10,1533.460500
20,869.673900
30,724.553100
40,711.363100
50,771.448700
60,871.737100
70,762.430100
80,656.884300
90,772.574100
100,726.574000


Eval Trained Model

In [ ]:
model = load_gliner_model("./models/gliner_uniner/checkpoint-100")
model.eval()

with torch.no_grad():
    metrics = model.evaluate(
        test_uniner_data,
        batch_size=8
    )

print(metrics)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


('P: 66.13%\tR: 33.92%\tF1: 44.84%\n', np.float64(0.44837127749924666))
